# Sentiment Analysis Model Training
This notebook trains the model, preserves important negative words, and saves the files required by the Streamlit app.

In [1]:
import re
import joblib
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("sentiment_dataset.csv")
print(df.shape)
df.head()

(500, 2)


,review,sentiment
0,Would not recommend this item. Review #112,Negative
1,"Five stars, completely satisfied. Review #74",Positive
2,Customer service was unhelpful. Review #125,Negative
3,Good value for money. Review #156,Positive
4,Excellent quality and fast delivery. Review #105,Positive


In [3]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\\breview\\s*#?\\s*\\d+\\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"[^a-z\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

df["clean_review"] = df["review"].apply(clean_text)
df[["review", "clean_review", "sentiment"]].head()

,review,clean_review,sentiment
0,Would not recommend this item. Review #112,would not recommend this item review,Negative
1,"Five stars, completely satisfied. Review #74",five stars completely satisfied review,Positive
2,Customer service was unhelpful. Review #125,customer service was unhelpful review,Negative
3,Good value for money. Review #156,good value for money review,Positive
4,Excellent quality and fast delivery. Review #105,excellent quality and fast delivery review,Positive


In [4]:
negative_words = {
    "no", "nor", "not", "never", "none", "nobody", "nothing",
    "neither", "nowhere", "cannot"
}
custom_stop_words = sorted(set(ENGLISH_STOP_WORDS) - negative_words)

print("not" in custom_stop_words)
print("never" in custom_stop_words)

False
False


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_review"], df["sentiment"],
    test_size=0.25, random_state=42, stratify=df["sentiment"]
)

tfidf = TfidfVectorizer(
    stop_words=custom_stop_words,
    ngram_range=(1, 2),
    min_df=1
)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

print("Train shape:", X_train_vec.shape)
print("Test shape:", X_test_vec.shape)

Train shape: (375, 110)
Test shape: (125, 110)


In [6]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_vec, y_train)

predictions = model.predict(X_test_vec)
print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

Accuracy: 1.0
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00        63
    Positive       1.00      1.00      1.00        62

    accuracy                           1.00       125
   macro avg       1.00      1.00      1.00       125
weighted avg       1.00      1.00      1.00       125



In [ ]:
joblib.dump(model, "sentiment_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

print("Model and TF-IDF vectorizer saved successfully.")

In [7]:
def predict_sentiment(review):
    cleaned = clean_text(review)
    vector = tfidf.transform([cleaned])
    prediction = model.predict(vector)[0]
    confidence = max(model.predict_proba(vector)[0]) * 100
    return prediction, confidence

tests = [
    "This product is excellent and amazing",
    "This product is bad",
    "This product is not good"
]

for text in tests:
    prediction, confidence = predict_sentiment(text)
    print(text, "->", prediction, f"({confidence:.1f}%)")

This product is excellent and amazing -> Positive (68.9%)
This product is bad -> Negative (71.5%)
This product is not good -> Negative (61.6%)


In [8]:
df

,review,sentiment,clean_review
0,Would not recommend this item. Review #112,Negative,would not recommend this item review
1,"Five stars, completely satisfied. Review #74",Positive,five stars completely satisfied review
2,Customer service was unhelpful. Review #125,Negative,customer service was unhelpful review
3,Good value for money. Review #156,Positive,good value for money review
4,Excellent quality and fast delivery. Review #105,Positive,excellent quality and fast delivery review
...,...,...,...
495,Good value for money. Review #107,Positive,good value for money review
496,Bad experience overall. Review #21,Negative,bad experience overall review
497,Very disappointed with the purchase. Review #99,Negative,very disappointed with the purchase review
498,The product stopped working quickly. Review #186,Negative,the product stopped working quickly review
